# Deploy Cross-Industry Pipeline to Fabric
### Programmatically create the Data Pipeline in your Fabric workspace

This notebook reads `CrossIndustry_Pipeline.json` and creates the Data Pipeline artifact in your current Fabric workspace using the REST API.

**What it does:**
1. Loads the pipeline JSON definition from the repo
2. Validates all referenced notebooks exist in the workspace
3. Creates or updates the Data Pipeline via Fabric REST API
4. Verifies successful deployment

> **Prerequisites:**
> - Run this notebook in a Fabric workspace (not local/VS Code)
> - All notebooks (00-07, ZT_Security_Utils, Pipeline_Logger) must be imported first
> - Workspace must have capacity assigned

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# STEP 1: VALIDATE FABRIC RUNTIME
# ════════════════════════════════════════════════════════════════════════

import json
import requests
import time

try:
    import sempy.fabric as fabric
    workspace_id = fabric.get_workspace_id()
    print(f"✓ Fabric runtime detected")
    print(f"  Workspace ID: {workspace_id}")
except Exception as e:
    raise RuntimeError(
        "This notebook must run inside a Fabric workspace. "
        "It cannot run locally or in VS Code. "
        f"Error: {e}"
    )

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# STEP 2: LOAD PIPELINE DEFINITION
# ════════════════════════════════════════════════════════════════════════

# Pipeline JSON is in repo root, one level up from cross_industry_notebooks/
PIPELINE_JSON_PATH = "/lakehouse/default/Files/CrossIndustry_Pipeline.json"

print(f"Loading pipeline definition from: {PIPELINE_JSON_PATH}")
print(f"\n⚠ NOTE: Upload CrossIndustry_Pipeline.json to your Lakehouse Files/ folder first!")
print(f"  Path: Files/CrossIndustry_Pipeline.json")

try:
    with open(PIPELINE_JSON_PATH, 'r') as f:
        pipeline_def = json.load(f)
    
    PIPELINE_NAME = pipeline_def.get("name", "CrossIndustry_Deployment_Pipeline")
    activities = pipeline_def.get("properties", {}).get("activities", [])
    
    print(f"\n✓ Pipeline definition loaded")
    print(f"  Name: {PIPELINE_NAME}")
    print(f"  Activities: {len(activities)} notebooks")
    for act in activities:
        print(f"    • {act['name']}")
        
except FileNotFoundError:
    print(f"\n✗ Pipeline JSON not found at: {PIPELINE_JSON_PATH}")
    print(f"\nTo fix:")
    print(f"  1. In your Fabric workspace, attach the Lakehouse to this notebook")
    print(f"  2. Upload CrossIndustry_Pipeline.json to Files/ in the Lakehouse")
    print(f"  3. Re-run this cell")
    raise

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# STEP 3: VALIDATE NOTEBOOKS EXIST IN WORKSPACE
# ════════════════════════════════════════════════════════════════════════

print("Validating all referenced notebooks exist in workspace...\n")

items_df = fabric.list_items()
notebooks_in_ws = set(items_df[items_df["Type"] == "Notebook"]["Display Name"].tolist())

required_notebooks = [
    "00_Industry_Config",
    "01_Data_Ingestion",
    "02_Load_Lakehouse",
    "03_Load_Warehouse",
    "04_Create_Semantic_Model",
    "05_Create_Ontology",
    "06_Create_Data_Agent",
    "07_Create_Dashboards",
    "ZT_Security_Utils",
    "Pipeline_Logger",
]

missing_notebooks = [nb for nb in required_notebooks if nb not in notebooks_in_ws]

if missing_notebooks:
    print(f"✗ Missing notebooks in workspace:")
    for nb in missing_notebooks:
        print(f"  • {nb}")
    print(f"\nImport these notebooks into your workspace before deploying the pipeline.")
    raise RuntimeError(f"Missing {len(missing_notebooks)} required notebook(s)")
else:
    print(f"✓ All {len(required_notebooks)} required notebooks found:")
    for nb in required_notebooks:
        print(f"  • {nb}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# STEP 4: CHECK IF PIPELINE ALREADY EXISTS
# ════════════════════════════════════════════════════════════════════════

items_df = fabric.list_items()
pipelines_in_ws = items_df[items_df["Type"] == "DataPipeline"]
existing_pipeline = pipelines_in_ws[pipelines_in_ws["Display Name"] == PIPELINE_NAME]

if not existing_pipeline.empty:
    existing_id = str(existing_pipeline.iloc[0]["Id"])
    print(f"⚠ Pipeline '{PIPELINE_NAME}' already exists")
    print(f"  Item ID: {existing_id}")
    print(f"\nOptions:")
    print(f"  1. Delete it manually in Fabric UI and re-run this notebook")
    print(f"  2. Set DELETE_EXISTING = True below to auto-delete")
    print(f"  3. Rename the pipeline in CrossIndustry_Pipeline.json")
    
    DELETE_EXISTING = False  # Set to True to auto-delete
    
    if DELETE_EXISTING:
        print(f"\n  Deleting existing pipeline...")
        access_token = notebookutils.credentials.getToken('pbi')
        resp = requests.delete(
            f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/items/{existing_id}",
            headers={"Authorization": f"Bearer {access_token}"},
            timeout=30
        )
        if resp.status_code in (200, 202, 204):
            print(f"  ✓ Deleted existing pipeline: {existing_id}")
            print(f"  Waiting 15s for deletion to propagate...")
            time.sleep(15)
        else:
            print(f"  ✗ Failed to delete: {resp.status_code} - {resp.text[:200]}")
            raise RuntimeError(f"Could not delete existing pipeline")
    else:
        raise RuntimeError(f"Pipeline '{PIPELINE_NAME}' already exists. See options above.")
else:
    print(f"✓ No existing pipeline named '{PIPELINE_NAME}'")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# STEP 5: RESOLVE NOTEBOOK REFERENCES
# ════════════════════════════════════════════════════════════════════════
# The pipeline JSON references notebooks by name. Fabric Data Pipeline
# activities need to use the correct notebook reference format.

print("Resolving notebook IDs for pipeline activities...\n")

items_df = fabric.list_items()
notebook_map = {}  # name → id

for _, row in items_df[items_df["Type"] == "Notebook"].iterrows():
    notebook_map[row["Display Name"]] = str(row["Id"])

print(f"Notebook ID mapping:")
for name in required_notebooks:
    nb_id = notebook_map.get(name, "NOT FOUND")
    print(f"  {name}: {nb_id[:8]}..." if nb_id != "NOT FOUND" else f"  {name}: {nb_id}")

# Update pipeline definition with notebook IDs
for activity in pipeline_def["properties"]["activities"]:
    notebook_name = activity["typeProperties"]["notebook"]["referenceName"]
    if notebook_name in notebook_map:
        # Keep the name reference - Fabric will resolve it
        print(f"  ✓ {activity['name']} → {notebook_name}")
    else:
        print(f"  ✗ {activity['name']} references missing notebook: {notebook_name}")
        raise RuntimeError(f"Notebook '{notebook_name}' not found in workspace")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# STEP 6: CREATE DATA PIPELINE VIA FABRIC REST API
# ════════════════════════════════════════════════════════════════════════

import base64

print(f"Creating Data Pipeline: {PIPELINE_NAME}...\n")

access_token = notebookutils.credentials.getToken('pbi')

# Encode the pipeline definition as base64 for the payload
pipeline_json_str = json.dumps(pipeline_def, indent=2)
pipeline_b64 = base64.b64encode(pipeline_json_str.encode('utf-8')).decode('utf-8')

# Create the DataPipeline item
payload = {
    "displayName": PIPELINE_NAME,
    "description": pipeline_def.get("properties", {}).get("description", 
        "Cross-Industry Accelerator deployment pipeline"),
    "type": "DataPipeline",
    "definition": {
        "parts": [
            {
                "path": "pipeline-content.json",
                "payload": pipeline_b64,
                "payloadType": "InlineBase64"
            }
        ]
    }
}

headers = {
    "Authorization": f"Bearer {access_token}",
    "Content-Type": "application/json"
}

url = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/items"

print(f"  API endpoint: {url}")
print(f"  Payload size: {len(pipeline_json_str):,} bytes")
print(f"  Activities: {len(activities)}")
print(f"\n  Sending request...")

resp = requests.post(url, headers=headers, json=payload, timeout=60)

if resp.status_code in (200, 201, 202):
    result = resp.json()
    pipeline_id = result.get("id", "N/A")
    
    print(f"\n✅ Data Pipeline created successfully!")
    print(f"  Name: {PIPELINE_NAME}")
    print(f"  Item ID: {pipeline_id}")
    print(f"  Workspace: {workspace_id}")
    
    if resp.status_code == 202:
        print(f"\n  ⏳ Pipeline is being provisioned (202 Accepted)")
        print(f"     It may take 30-60 seconds to appear in the workspace.")
    
    print(f"\n  Next steps:")
    print(f"    1. Go to your Fabric workspace → Data Factory section")
    print(f"    2. Find '{PIPELINE_NAME}'")
    print(f"    3. Click 'Run' to trigger the pipeline")
    print(f"    4. Monitor progress in the Monitoring hub")
    print(f"    5. Query audit logs: SELECT * FROM dbo.pipeline_runs")
    
else:
    print(f"\n✗ Failed to create pipeline")
    print(f"  Status code: {resp.status_code}")
    print(f"  Response: {resp.text[:1000]}")
    
    # Common error scenarios
    if resp.status_code == 400:
        print(f"\n  Possible causes:")
        print(f"    • Invalid pipeline definition format")
        print(f"    • Referenced notebooks don't exist")
        print(f"    • Workspace doesn't support Data Pipelines")
    elif resp.status_code == 401:
        print(f"\n  Authentication failed. Try re-running the notebook.")
    elif resp.status_code == 403:
        print(f"\n  Insufficient permissions. You need Contributor or Admin role.")
    
    raise RuntimeError(f"Pipeline creation failed: {resp.status_code}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# STEP 7: VERIFY DEPLOYMENT
# ════════════════════════════════════════════════════════════════════════

print("Verifying pipeline deployment...\n")

# Wait a few seconds for the item to be indexed
time.sleep(5)

items_df = fabric.list_items()
pipelines = items_df[items_df["Type"] == "DataPipeline"]
our_pipeline = pipelines[pipelines["Display Name"] == PIPELINE_NAME]

if not our_pipeline.empty:
    pipeline_info = our_pipeline.iloc[0]
    print(f"✓ Pipeline verified in workspace:")
    print(f"  Name: {pipeline_info['Display Name']}")
    print(f"  ID: {pipeline_info['Id']}")
    print(f"  Type: {pipeline_info['Type']}")
    print(f"\n  Pipeline activities that will execute:")
    for act in activities:
        deps = act.get('dependsOn', [])
        dep_str = f" (depends on: {', '.join(d['activity'] for d in deps)})" if deps else " (starts immediately)"
        print(f"    • {act['name']}{dep_str}")
else:
    print(f"⚠ Pipeline not found in workspace list (may still be provisioning)")
    print(f"  Check the workspace in 30-60 seconds.")

print(f"\n{'═'*70}")
print(f"DEPLOYMENT COMPLETE")
print(f"{'═'*70}")
print(f"\n🚀 Your Cross-Industry Accelerator pipeline is ready!")
print(f"\n   Go to: Fabric workspace → Data Factory → {PIPELINE_NAME}")
print(f"   Click: Run")
print(f"\n{'═'*70}")